# Module 8 – Artificial Neural Networks (ANN)
## In‑Class Practice Notebook

**Goal:** Explore and compare two neural network models:

1. A **Multi‑Layer Perceptron (MLP)** implemented with `MLPClassifier` in scikit‑learn.
2. A simple **RBF network** built using **K‑Means + Ridge Regression**.

You will:
- Load and inspect a real medical dataset (breast cancer classification).
- Train and evaluate an MLP model with different architectures.
- Build an RBF network and compare it to the MLP.
- Reflect on when MLP vs RBF might be preferable.

> **Reminder:** Run the cells from top to bottom. Read the comments carefully – several cells contain **`Students To Do Activity`** instructions.


In [ ]:
# ==============================
# 1. Imports & Global Settings
# ==============================
# We import all required libraries here so that later cells focus
# only on modeling and interpretation.

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# For reproducibility: using a fixed random state helps us get the
# same train/test split and model behavior each time we run the notebook.
RANDOM_STATE = 42

# Make printed numpy arrays easier to read
np.set_printoptions(precision=3, suppress=True)

print("Libraries imported successfully.")

In [ ]:
# =======================================
# 2. Load & Inspect the Breast Cancer Data
# =======================================
# We use the classic Breast Cancer Wisconsin dataset from scikit-learn.
# This is a binary classification problem: malignant vs benign tumor.

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

print("Dataset shape (samples, features):", X.shape)
print("Target classes:", target_names)
print("Class distribution (counts):", np.bincount(y))
print("First 5 feature names:", feature_names[:5])

# ===============================
# Students To Do Activity (1)
# ===============================
# 1. Print the first 3 rows of X to see raw feature values.
# 2. Print the min/max of one feature (e.g., mean radius).
#    Hint: use X[:, 0].min() and X[:, 0].max().

# Uncomment and complete the following lines if you want to try:
# print("First 3 rows of X:\n", X[:3])
# print("Min / Max of first feature:", X[:, 0].min(), X[:, 0].max())

In [ ]:
# =======================================
# 3. Train / Test Split + Feature Scaling
# =======================================
# Why scaling?
# ------------
# For many models (including MLP and methods using Euclidean distance),
# it is important that features are on a similar scale. Otherwise,
# features with large numeric ranges dominate the distance / gradient
# calculations and can hurt learning.
#
# We use StandardScaler to transform each feature to have:
#   - mean ≈ 0
#   - standard deviation ≈ 1

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,            # 20% for testing
    random_state=RANDOM_STATE,
    stratify=y               # preserve class ratio in train & test
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape     :", X_test_scaled.shape)

# ===============================
# Students To Do Activity (2)
# ===============================
# 1. Try changing test_size to 0.3 and re-run from this cell downwards.
#    Observe how it affects accuracy and stability of results.
# 2. (Optional) Print the mean and std of the scaled features to verify
#    that scaling worked as expected.

# Example (you can uncomment):
# print("Mean of first feature (train, scaled):", X_train_scaled[:, 0].mean())
# print("Std of first feature  (train, scaled):", X_train_scaled[:, 0].std())

In [ ]:
# =======================================
# 4. MLP Classifier – Baseline Model
# =======================================
# We start with a simple MLP architecture:
#   - Two hidden layers: (32, 16)
#   - Activation: ReLU
#   - Optimizer (solver): Adam
#
# Why MLPClassifier?
# ------------------
# scikit-learn's MLPClassifier implements a standard feedforward
# neural network trained with backpropagation. It handles the
# training loop, gradients, and updates internally, so we can
# focus on architecture and hyperparameters.

hidden_layers = (32, 16)
activation = "relu"

mlp = MLPClassifier(
    hidden_layer_sizes=hidden_layers,
    activation=activation,
    solver="adam",
    max_iter=300,
    random_state=RANDOM_STATE,
)

print("Training baseline MLP with layers =", hidden_layers, "and activation =", activation)
mlp.fit(X_train_scaled, y_train)

y_pred_mlp = mlp.predict(X_test_scaled)

print("\n=== Baseline MLP Results ===")
print("Test Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred_mlp)))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

# ===============================
# Students To Do Activity (3)
# ===============================
# 1. Change activation from 'relu' to 'tanh' or 'logistic'.
# 2. Try a different optimizer/solver: 'sgd' or 'lbfgs'.
# 3. Observe:
#    - Does training converge (any warnings)?
#    - How does test accuracy change?
#    - Does the confusion matrix pattern change significantly?

In [ ]:
# =======================================================
# 5. Experiment: Different MLP Depths / Architectures
# =======================================================
# Here we compare several architectures with increasing depth / width.
# Idea: Deeper / wider networks have more capacity, but may overfit or
# be harder to train on small datasets.

architectures = [
    (16,),                # 1 hidden layer, 16 units
    (32, 16),             # 2 hidden layers (baseline)
    (64, 32, 16),         # 3 hidden layers
    (64, 64, 32, 16),     # 4 hidden layers
]

results_mlp = []

for arch in architectures:
    model = MLPClassifier(
        hidden_layer_sizes=arch,
        activation="relu",
        solver="adam",
        max_iter=400,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results_mlp.append((arch, acc))

print("=== MLP Architecture Comparison (ReLU, Adam) ===")
for arch, acc in results_mlp:
    print(f"Architecture {arch:20s} -> Test Accuracy = {acc:.3f}")

# ===============================
# Students To Do Activity (4)
# ===============================
# 1. Add more architectures into the list above (e.g., 7-layer network).
# 2. Observe whether accuracy always improves with more layers.
# 3. Think: On a small tabular dataset, is a *very* deep network helpful
#    or does it simply make training slower / less stable?

In [ ]:
# =======================================
# 6. RBF Network – Concept & Implementation
# =======================================
# RBF networks use *localized* activation functions (typically Gaussians)
# centered at specific points in feature space.
#
# Architecture here:
#   1) Use K-Means to choose RBF centers (unsupervised step).
#   2) Compute RBF activations (Gaussian kernel) for each sample.
#   3) Train a simple linear model (Ridge regression) on top of these
#      RBF features for classification.
#
# This is a two-stage training approach, unlike MLP's end-to-end backprop.

def rbf_kernel(X, centers, gamma):
    """Compute Gaussian RBF features.
    
    Parameters
    ----------
    X : array, shape (n_samples, n_features)
        Input data.
    centers : array, shape (n_centers, n_features)
        RBF centers (here from K-Means).
    gamma : float
        Controls the width of the Gaussian. Larger gamma = narrower peaks.
    """
    dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    return np.exp(-gamma * dists**2)

# Choose the number of RBF centers.
n_centers = 25

print("Fitting K-Means with n_centers =", n_centers)
kmeans = KMeans(n_clusters=n_centers, random_state=RANDOM_STATE)
kmeans.fit(X_train_scaled)
centers = kmeans.cluster_centers_

# Heuristic for gamma: inverse of (2 * variance)
gamma = 1.0 / (2 * np.var(X_train_scaled))
print("Gamma used for RBF =", gamma)

Phi_train = rbf_kernel(X_train_scaled, centers, gamma)
Phi_test = rbf_kernel(X_test_scaled, centers, gamma)

print("RBF feature matrix shape (train):", Phi_train.shape)

# Train linear model on top of RBF features
rbf_model = Ridge(alpha=1.0)
rbf_model.fit(Phi_train, y_train)

y_scores_rbf = rbf_model.predict(Phi_test)
y_pred_rbf = (y_scores_rbf >= 0.5).astype(int)

print("\n=== RBF Network Results ===")
print("Test Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred_rbf)))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rbf, target_names=target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rbf))

# ===============================
# Students To Do Activity (5)
# ===============================
# 1. Change n_centers (e.g., 10, 40, 60) and re-run this cell.
# 2. Observe how test accuracy changes with very few vs many centers.
# 3. Discuss: How does the choice of gamma influence underfitting vs
#    overfitting in an RBF network?

In [ ]:
# =======================================
# 7. Summary & Reflection
# =======================================
print("MLP baseline accuracy : {:.3f}".format(accuracy_score(y_test, y_pred_mlp)))
print("RBF network accuracy  : {:.3f}".format(accuracy_score(y_test, y_pred_rbf)))

print("\nQuestions for Reflection:")
print("1) Which model performed better on this dataset: MLP or RBF?")
print("2) How sensitive was the MLP to changes in depth (number of layers)?")
print("3) How sensitive was the RBF network to n_centers and gamma?")
print("4) In a real healthcare application, what other factors (besides raw accuracy)")
print("   would matter when choosing between MLP and RBF (e.g., interpretability,")
print("   training time, hardware constraints)?")

# ===============================
# Students To Do Activity (6)
# ===============================
# Write a short (5–7 sentence) reflection in a separate cell or Word
# document answering the questions above. Try to connect your answers
# to both the *theory* from the slides and the *results* you observed here.